In [ ]:
import csv
import statistics

import rospy
from std_msgs.msg import Float32MultiArray, Int8MultiArray

GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
RESET_ALL = "\033[0m"


class MonitorArmLatency:
    def __init__(self, num_trials=100, output_csv="arm_latency_log.csv"):
        rospy.init_node('monitor_arm_latency',
                        anonymous=True, disable_signals=True)
        self._joints_sub = rospy.Subscriber(
            '/ugv0/telehandler/joints', Float32MultiArray, self.joints_callback)
        self._telejoy_pub = rospy.Publisher(
            '/ugv0/telehandler/telejoy', Int8MultiArray, queue_size=10)

        self.num_trials = num_trials
        self.output_csv = output_csv
        self.latencies = []
        self.current_pub_time = None
        self.trial_active = False
        self.start_position = []

    def joints_callback(self, msg):
        if self.trial_active:
            # On first callback after publish, record start state
            if not self.start_position:
                self.start_position = list(msg.data)
                return
            # On subsequent callback, detect movement and record latency
            if list(msg.data) != self.start_position:
                t_sub = rospy.Time.now()
                latency = (t_sub - self.current_pub_time).to_sec()
                self.latencies.append(latency)
                rospy.loginfo(
                    f"Trial {len(self.latencies)} latency: {latency:.4f} s")
                self.trial_active = False  # end this trial

    def publish_telejoy(self, teleop_values):
        # Stamp publish time and send command
        self.current_pub_time = rospy.Time.now()
        telejoy_msg = Int8MultiArray(data=teleop_values)
        self._telejoy_pub.publish(telejoy_msg)

    def measure_latency(self, teleop_values  = None):
        if teleop_values is None:
            teleop_values = [1, 0, 0, 0, 0, 0]  # example movement on joint 0

        rospy.sleep(1.0)  # allow some time for ROS topics to warm up
        rate = rospy.Rate(20)
        

        for _ in range(self.num_trials):
            # Prepare for a new trial
            self.start_position = []
            self.trial_active = True

            # Publish command
            self.publish_telejoy(teleop_values)

            # Wait until latency is recorded or timeout
            start_loop = rospy.Time.now()
            while self.trial_active and (rospy.Time.now() - start_loop).to_sec() < 0.2:
                rate.sleep()

        # After trials, write CSV and print stats
        with open(self.output_csv, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['trial', 'latency_ms'])
            for idx, lat in enumerate(self.latencies, start=1):
                writer.writerow([idx, f"{lat * 1000:.3f}"])
            writer.writerow(['mean', f"{statistics.mean(self.latencies) * 1000:.3f}"])
            writer.writerow(['max', f"{max(self.latencies) * 1000:.3f}"])
            writer.writerow(['stddev', f"{statistics.stdev(self.latencies) * 1000:.3f}" if len(self.latencies) > 1 else 0.0])

        mean_lat = statistics.mean(self.latencies) * 1000
        max_lat = max(self.latencies) * 1000
        stddev_lat = (statistics.stdev(self.latencies) if len(self.latencies) > 1 else 0.0) * 1000

        rospy.loginfo(f"\n--- Latency Results ---")
        rospy.loginfo(f"Trials: {len(self.latencies)}")
        rospy.loginfo(f"Mean: {mean_lat:.1f} ms")
        rospy.loginfo(f"Max: {max_lat:.1f} ms")
        rospy.loginfo(f"Std Dev: {stddev_lat:.1f} ms")

In [18]:
move_arm = [0, 0, 0, 80, 0, 0]
monitor = MonitorArmLatency(num_trials=100, output_csv="arm_latency_log.csv")
monitor.measure_latency(move_arm)

[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 1 latency: 0.1861 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 2 latency: 0.1867 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 3 latency: 0.1845 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 4 latency: 0.1850 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 5 latency: 0.1865 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 6 latency: 0.1856 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 7 latency: 0.1866 s
[INFO] [/monitor_arm_latency_117705_1753958004678] [MonitorArmLatency.joints_callback] [40]: Trial 8 latency: 0.1873 s
[INFO] [/monitor_arm_latency_117705_175395800467

KeyboardInterrupt: 